# Andreatta Fold-Change Comparison Dashboard
**Project:** Andreatta et al. 2026 Replication & Sub-Categorical Filtering Workspace
**Authors:** Alina Krawczynski, Rannyele Passos Ribeiro (powered by Claude)
**Lab:** Özpolat Lab, Washington University in St. Louis, 1 Brookings Drive, St. Louis, MO 63130
**Year:** 2026

---

## Dataset Overview & Biological Context
This study has **no single expression matrix** to slice by condition. Its transcriptomics results
(Andreatta et al. 2026, PNAS 123(12):e2514719123, `doi:10.1073/pnas.2514719123`) are ~20 separate
**pairwise differential-expression (DE) comparisons** between specific *Platynereis dumerilii* genotypes
(l-cry⁻ᐟ⁻, nr0b1/2⁻ᐟ⁻, nr0b1/2⁺ᐟ⁻, and their respective wild types), each mapped onto either the draft
**genome** (`XLOC_...` IDs) or the **reference transcriptome** from Häfker et al. 2024 (`comp..._c#_seq#`
IDs).

So instead of "expression of gene X across conditions", this dashboard shows **log2 fold change of gene X
across whichever pairwise comparisons it was tested in** — bars above zero mean upregulated in the mutant,
below zero means downregulated, colored by statistical significance (padj < 0.05).

**On normalization:** unlike the other dashboards in this series, this one doesn't show absolute expression
at all, so there's no normalization step performed here. Every log2FC/padj value plotted is read directly
from the paper's own precomputed differential-expression tables (Datasets S1/S3/S4), which the paper
generated with its own upstream normalization and DE-testing pipeline — see the paper's Methods for that
procedure.

It reads the already-cleaned tables produced by `Data Cleaning Analysis.ipynb` (in `cleaned/`) rather than
re-parsing the raw supplementary Excel files, so run that notebook first if `cleaned/` is missing.

### Recap of data quirks already handled upstream
* Dataset S3A's ~66,745 exact-duplicate rows were removed (see the cleaning notebook for details).
* Annotation strings were parsed into a clean best-hit description from the compound BLAST format.
* Dataset S5's curated gene-of-interest list (with clean, human-readable gene symbols like `cp17a1/2`,
  `dhr11`) is used here as a symbol-search shortcut.

---

## Key Features
* **Dual Search Modes:** search by raw ID (`XLOC_059112`, `comp2263280_c0_seq1`) *or* by a curated gene
  symbol (marked with a ★ in the picker, e.g. `cp17a1/2`) — symbols expand to every matching gene ID
  automatically.
* **Two-Level Comparison Filtering:** pick one genotype comparison in the **Pairwise** dropdown
  (l-cry⁻ᐟ⁻ vs l-cry⁺ᐟ⁺, or one of the nr0b1/2 comparisons), then hold `Ctrl`/`Cmd` in the **Details**
  box to pick any subset of *that* comparison's actual tissue/mapping/DE-analysis variants — the Details
  list updates automatically to only show combinations that really exist for the chosen comparison.
* **Dual-Source Annotation:** each searched gene is checked against the curated gene-of-interest summary
  first, then the DE tables' own parsed BLAST best-hit description as a fallback.
* **Significance-Colored Bars:** red = significant upregulation, blue = significant downregulation, gray =
  not significant (padj ≥ 0.05); comparisons the gene wasn't tested in are simply omitted from the plot (not
  shown as a zero bar, since zero would misleadingly read as "no change").
* **Abbreviated Chart Labels + Key:** chart x-axes use short codes (e.g. `S3-ImmH-G-DE1`) instead of the
  full comparison text, which was too long to fit on the page — axes are abbreviated and a key spelling
  out every code is printed above every batch of charts, with the full, unabbreviated comparison name
  always still shown in the printed status table.
* **On-Demand Exporter:** writes vector `.pdf` plots and a combined annotation/comparison-status `.csv` to
  `exported_results/`.
* **Multi-Gene Heatmap & Dot Plot:** Selecting 2+ genes automatically adds a genes-by-comparison log2FC heatmap and dot plot (blank cells = not tested, dot size = significance) after the individual bar charts, included in the PDF export.

### Why do so many comparisons show "Not tested"?
This is expected, not a bug — two real reasons: (1) genome-mapped (`XLOC_...`) and transcriptome-mapped
(`comp...`) genes are two entirely separate ID spaces, so a genome gene will never have data in a
transcriptome-mapped sheet and vice versa; (2) Dataset S1 only covers a small curated subset of genes for
specific figure panels, while S3/S4 are much larger, so a gene absent from S1 may still be well-tested
elsewhere. "Not tested" rows are always excluded from the bar chart itself; they only appear in the printed
status table, shown as `-` rather than a raw `NaN`.

---

## Dashboard User Guide

### Step 1: Choose a Pairwise Comparison, then its Details
Pick one genotype comparison from the **Pairwise** dropdown. The **Details** box below it refreshes to show
only that comparison's real tissue/mapping/DE-analysis variants (2–10 options depending on the comparison);
hold `Ctrl` (or `Cmd` on Mac) to pick any subset — all are selected by default.

### Step 2: Choose Target Genes (Use A, B, or Both!)
* **Option A (Filter List):** type into **Filter List** to narrow the picker, then click (or Ctrl/Cmd-click)
  IDs or ★-marked symbols.
* **Option B (Paste Box):** paste raw IDs or gene symbols, separated by commas or spaces.

### Step 3: Run and/or Save
* Click **🎯 Generate Visuals (View Only)** to render fold-change bar charts, annotation tables, and
  per-comparison significance status instantly on-screen.
* Click **💾 Export Active Results (Save CSV/PDF)** to save vector assets to `exported_results/`.


In [ ]:
# ============================================================================================
#  DEPENDENCY CHECK -- run this cell first.
#  Installs any Python packages this notebook needs that aren't already present, then you're
#  done. Safe to re-run (does nothing if everything is already installed). To install by hand
#  instead, run this in a terminal / Anaconda Prompt:
#      pip install pandas numpy matplotlib seaborn scipy ipywidgets
# ============================================================================================
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = ['pandas', 'numpy', 'matplotlib', 'seaborn', 'scipy', 'ipywidgets']

_missing = [p for p in REQUIRED_PACKAGES if importlib.util.find_spec(p) is None]
if _missing:
    print("Installing missing packages:", ", ".join(_missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *_missing])
    print("\nInstall finished. If an import fails in a later cell, restart the kernel")
    print("(menu: Kernel -> Restart Kernel) and run the notebook again from the top.")
else:
    print("All required packages are already installed -- you're good to go.")

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# This packaged copy keeps all input files flat alongside the notebook (no cleaned/ subfolder --
# that layout was specific to the working repo, where this data was produced by a separate
# upstream cleaning notebook).

# 1. Load the combined DE-results table (Datasets S1, S3, S4) and the curated gene-of-interest summary.
# Defensively drop any row with a missing gene ID here too (not just upstream in the cleaning notebook):
# pandas' newer nullable string dtype does not coerce a missing value into the text "nan" the way older
# pandas versions did, so a stray blank gene_id would otherwise crash sorted() below with
# "'<' not supported between instances of 'float' and 'str'".
df_de = pd.read_csv("de_results_all_comparisons.csv", low_memory=False)
df_de = df_de.dropna(subset=["gene_id"])
df_de["gene_id"] = df_de["gene_id"].astype(str)

df_summary = pd.read_csv("curated_gene_summary.csv")
df_summary = df_summary.dropna(subset=["gene_id"])
df_summary["gene_id"] = df_summary["gene_id"].astype(str)

# 2. Build one readable label per pairwise comparison. (dataset, comparison, tissue, mapping, de_analysis)
# together uniquely identify one of the 20 DE sheets, so this label doubles as a unique comparison key.
# Built with vectorized string concatenation rather than df.apply(axis=1) -- row-wise apply calls a
# Python function once per row (130,914 rows here), which measured ~1.8s per column; the vectorized
# form below does the equivalent work in well under a second by operating on whole columns at once.
def _de_suffix(df, sep_fmt):
    """Vectorized equivalent of `f' | DE analysis {n}'` / `f'-DE{n}'` appended only where
    de_analysis is present -- avoids a per-row Python call for the common case."""
    valid = df["de_analysis"].notna()
    suffix = pd.Series("", index=df.index)
    suffix.loc[valid] = df.loc[valid, "de_analysis"].astype(int).astype(str).apply(sep_fmt.format)
    return suffix

_de_suffix_long = _de_suffix(df_de, " | DE analysis {}")
_de_suffix_short = _de_suffix(df_de, "-DE{}")

df_de["comparison_label"] = (
    df_de["dataset"].astype(str) + " | " + df_de["comparison"].astype(str) + " | " +
    df_de["tissue"].astype(str) + " | " + df_de["mapping"].astype(str) + _de_suffix_long
)
COMPARISON_OPTIONS = sorted(df_de["comparison_label"].unique())

# 2b. Split that same label into two levels for the picker widget: a "Pairwise" genotype comparison
# (4 options: lcry-/- vs lcry+/+, nr0b+/- vs nr0b+/+, nr0b-/- vs nr0b+/+, nr0b-/- vs nr0b+/-) and,
# within it, a "Details" level (dataset, tissue, mapping, DE analysis) -- since not every detail
# combination exists for every genotype comparison (e.g. "premature female heads" only occurs for the
# lcry comparison), the details list is rebuilt per comparison rather than offered as one flat 20-item
# multiselect.
df_de["detail_label"] = (
    df_de["dataset"].astype(str) + " | " + df_de["tissue"].astype(str) + " | " +
    df_de["mapping"].astype(str) + _de_suffix_long
)

COMPARISON_TYPES = sorted(df_de["comparison"].unique())
COMPARISON_DETAILS = {}
for comp_type in COMPARISON_TYPES:
    sub = df_de.loc[df_de["comparison"] == comp_type, ["detail_label", "comparison_label"]].drop_duplicates()
    COMPARISON_DETAILS[comp_type] = dict(zip(sub["detail_label"], sub["comparison_label"]))

# 2c. Short abbreviated codes for plot x-axis labels (the full labels above are too long to fit on a
# chart), plus a lookup back to the full comparison type and a printable legend explaining each code.
TISSUE_ABBR = {"immature heads": "ImmH", "premature female heads": "PrFH", "immature whole worms": "ImmW"}
MAPPING_ABBR = {"genome": "G", "transcriptome": "T"}
COMPARISON_ABBR = {
    "lcry-/- vs lcry+/+": "lcry",
    "nr0b+/- vs nr0b+/+": "nr0bHet",
    "nr0b-/- vs nr0b+/+": "nr0bHom",
    "nr0b-/- vs nr0b+/-": "nr0bHomHet",
}

_tissue_abbr_col = df_de["tissue"].map(TISSUE_ABBR).fillna(df_de["tissue"])
_mapping_abbr_col = df_de["mapping"].map(MAPPING_ABBR).fillna(df_de["mapping"])
df_de["detail_abbr"] = df_de["dataset"].astype(str) + "-" + _tissue_abbr_col + "-" + _mapping_abbr_col + _de_suffix_short

LABEL_TO_ABBR = dict(zip(df_de["comparison_label"], df_de["detail_abbr"]))
LABEL_TO_COMPARISON = dict(zip(df_de["comparison_label"], df_de["comparison"]))

ABBREVIATION_LEGEND = pd.DataFrame([
    {"Code": "S1 / S3 / S4", "Meaning": "Source dataset (PNAS Dataset S1, S3 or S4)"},
    {"Code": "ImmH / PrFH / ImmW", "Meaning": "immature heads / premature female heads / immature whole worms"},
    {"Code": "G / T", "Meaning": "reads mapped on the draft Genome / reference Transcriptome"},
    {"Code": "DE1 / DE2", "Meaning": "1st / 2nd independent differential-expression analysis"},
] + [{"Code": abbr, "Meaning": f'Pairwise comparison: "{full}"'} for full, abbr in COMPARISON_ABBR.items()])

# 3. Map each curated gene SYMBOL to its underlying gene ID(s), enabling symbol-based search
SYMBOL_TO_IDS = df_summary.groupby("gene name")["gene_id"].apply(lambda s: sorted(set(s))).to_dict()

VALID_GENE_IDS = set(df_de["gene_id"])

# 4. Index (gene_id, comparison_label) -> {log2FoldChange, padj} once, as a plain dict. Every gene
# lookup used to re-scan this whole 130K-row table with a fresh boolean mask (df_de[(df_de["gene_id"]
# == gene_id) & (df_de["comparison_label"] == label)]) -- fine for one gene, but _comparison_table is
# called up to 4x per gene per Submit/Export (heatmap build, bar charts, annotation print, export), so
# a 30-gene keyword-search result could re-scan the table ~2,400 times (measured: ~58s worst case).
# A dict lookup here is O(1) instead of O(130,914) per (gene, comparison) pair.
GENE_COMPARISON_INDEX = df_de.set_index(["gene_id", "comparison_label"])[["log2FoldChange", "padj", "baseMean"]].to_dict("index")

n_summary_ids = df_summary["gene_id"].nunique()
print(f"✅ Loaded {len(df_de)} DE result rows across {len(COMPARISON_OPTIONS)} pairwise comparisons.")
print(f"✅ Curated gene-of-interest summary: {n_summary_ids} unique gene IDs / "
      f"{len(SYMBOL_TO_IDS)} gene symbols available for search-by-symbol.")


---

## 🔬 BLAST Sequence Search
Have a nucleotide sequence from elsewhere and want to know if/where it shows up in this dataset? Paste it
below to BLAST it against the **HAMN01 reference transcriptome** (ENA accession `HAMN01` — a local
`blastn` search, no internet required). Any matching `comp..._c#_seq#` transcript ID found can then be
copied into the **Paste Box** in Step 2 above to view that gene's fold-change profile.

**Important limitation:** this only covers **transcriptome-mapped** genes (`comp..._c#_seq#` IDs). The
**genome-mapped** genes (`XLOC_...` IDs) in this dashboard were quantified against a draft genome assembly,
not this transcriptome, and no matching genome reference sequence is currently available in this project —
so a sequence search cannot identify `XLOC_...` genes. A hit here will always be a `comp...`-style ID.

**Setup (optional — the rest of the dashboard works without this):** the sequence search needs
the free NCBI BLAST+ toolkit plus a small search database you build once from the reference
transcriptome. Step-by-step instructions — install BLAST+, get the reference FASTA, and build
the database into a `blast_db/` folder next to this notebook — are in the comment at the top of
the next code cell. The notebook finds BLAST+ automatically once it is installed; until then this
section just prints a short reminder and everything else in the notebook is unaffected.

In [ ]:
import shutil
import subprocess
import tempfile
import re

# --- Sequence -> Gene ID search: paste any raw nucleotide sequence and find the best-matching
# transcript(s) in the HAMN01 reference transcriptome via a local blastn search. Covers only
# transcriptome-mapped genes (comp..._c#_seq# IDs) -- see the markdown note above for why
# genome-mapped (XLOC_...) genes aren't searchable this way.
#
# SETUP (only needed once per machine): install BLAST+ from
# https://ftp.ncbi.nlm.nih.gov/blast/executables/blast+/ , download the HAMN01 assembly:
#   curl -o HAMN01.fasta.gz https://ftp.ebi.ac.uk/pub/databases/ena/tsa/public/ham/HAMN01.fasta.gz
#   gunzip HAMN01.fasta.gz
# then rewrite headers so the comp..._c#_seq# transcript ID is the first token (required for
# ID-based lookup/export, since the original "ENA|...|..." header style isn't a BLAST-recognized
# accession format):
#   python -c "
#   import re
#   with open('HAMN01.fasta') as fin, open('renamed.fasta', 'w') as fout:
#       for line in fin:
#           if line.startswith('>'):
#               m = re.search(r'(comp\d+_c\d+_seq\d+)', line)
#               fout.write(f'>{m.group(1)} {line[1:].strip()}\n' if m else line)
#           else:
#               fout.write(line)
#   "
# then build the database with -parse_seqids so sequences can be retrieved by that ID:
#   makeblastdb -in renamed.fasta -dbtype nucl -parse_seqids -out blast_db/andreatta_transcriptome
# The notebook auto-detects BLAST+ on your PATH (or set BLAST_BIN_DIR
# below) and looks for the database in ./blast_db/.
# Once built, the database is fully self-contained -- the original HAMN01 FASTA is not needed for
# searching OR sequence export, and can be deleted to save space if desired.

# --- BLAST+ location (auto-detected) ---------------------------------------------------------
# By default this notebook finds the BLAST+ programs (blastn, blastdbcmd, makeblastdb) on your
# system PATH -- if you installed BLAST+ the normal way, leave BLAST_BIN_DIR = "" and it just
# works. If BLAST+ lives somewhere not on your PATH, set BLAST_BIN_DIR to the folder that holds
# the blastn / blastdbcmd / makeblastdb executables.
BLAST_BIN_DIR = ""   # e.g. r"C:\Program Files\NCBI\blast-2.17.0+\bin"  or  "/usr/local/ncbi/blast/bin"

# The prebuilt search database: a set of files sharing one name prefix (<name>.nin, .nhr, .nsq,
# ...). Defaults to a "blast_db" folder next to this notebook. Build it once from the reference
# transcriptome -- see the SETUP note above. This whole BLAST section is optional; the rest of
# the dashboard works fine without it.
BLAST_DB_PATH = os.path.join("blast_db", "andreatta_transcriptome")

def _find_blast_tool(tool_name):
    """Locate a BLAST+ executable: honour BLAST_BIN_DIR if it is set, otherwise search the
    system PATH. Returns the full path or None. shutil.which adds the right extension (.exe)
    on Windows automatically."""
    if BLAST_BIN_DIR:
        found = shutil.which(tool_name, path=BLAST_BIN_DIR)
        if found:
            return found
    return shutil.which(tool_name)

# Cache of the most recent search, so the export button can act on whatever was last searched
_CURRENT_BLAST_QUERY = None
_CURRENT_BLAST_HITS = None

def _to_fasta(raw_text):
    """Wrap raw pasted text into a single-record FASTA query -- accepts either bare sequence
    (any whitespace/line breaks) or an already-FASTA-formatted block with its own header."""
    raw_text = raw_text.strip()
    if raw_text.startswith(">"):
        return raw_text
    seq = "".join(raw_text.split())
    return f">user_query\n{seq}"

def run_blast_search(raw_sequence, max_hits=10):
    """BLASTs a pasted nucleotide sequence against the HAMN01 reference transcriptome and returns
    a results DataFrame with one row per hit. Gene_ID is the raw comp..._c#_seq# transcript ID
    (no isoform suffix to strip here -- this dashboard keys directly on the transcript ID)."""
    blastn_exe = _find_blast_tool("blastn")
    if blastn_exe is None:
        print("❌ BLAST+ not found on your system. Install it (see the SETUP note above) or set")
        print("   BLAST_BIN_DIR at the top of this cell. The rest of the dashboard works without BLAST.")
        return None
    if not os.path.isfile(BLAST_DB_PATH + ".nin"):
        print(f"❌ BLAST database not found at {BLAST_DB_PATH} -- see the setup note above this cell.")
        return None

    fasta_query = _to_fasta(raw_sequence)
    seq_only = "".join(line for line in fasta_query.splitlines() if not line.startswith(">"))
    if len(seq_only) < 15:
        print("❌ Query sequence looks too short (or empty) to BLAST -- paste at least ~15 bases.")
        return None

    with tempfile.NamedTemporaryFile(mode="w", suffix=".fasta", delete=False) as f:
        f.write(fasta_query)
        query_path = f.name

    try:
        result = subprocess.run(
            [blastn_exe, "-query", query_path, "-db", BLAST_DB_PATH,
             "-outfmt", "6 sseqid pident length evalue bitscore stitle",
             "-max_target_seqs", str(max_hits)],
            capture_output=True, text=True, timeout=60
        )
    finally:
        os.remove(query_path)

    if result.returncode != 0:
        print("❌ blastn reported an error:")
        print(result.stderr.strip())
        return None

    lines = [l for l in result.stdout.strip().splitlines() if l.strip()]
    if not lines:
        print("🛑 No BLAST hits found for this sequence in the HAMN01 transcriptome.")
        return None

    rows = []
    for line in lines:
        sseqid, pident, length, evalue, bitscore, stitle = line.split("\t", 5)
        rows.append({
            "Gene_ID": sseqid,
            "In_Dashboard": sseqid in VALID_GENE_IDS,
            "Transcript_Hit": sseqid,
            "%Identity": float(pident),
            "Alignment_Length": int(length),
            "E-value": float(evalue),
            "Bit_Score": float(bitscore),
        })

    df_hits = pd.DataFrame(rows).sort_values("Bit_Score", ascending=False).reset_index(drop=True)
    return df_hits

def get_sequence_from_db(seq_id):
    """Retrieves a sequence's full FASTA record directly from the BLAST database via blastdbcmd
    -- works entirely from the compiled database, without needing the original HAMN01 FASTA on
    disk."""
    blastdbcmd_exe = _find_blast_tool("blastdbcmd")
    if blastdbcmd_exe is None:
        print("❌ BLAST+ not found on your system. Install it (see the SETUP note above) or set")
        print("   BLAST_BIN_DIR at the top of this cell. The rest of the dashboard works without BLAST.")
        return None
    result = subprocess.run(
        [blastdbcmd_exe, "-db", BLAST_DB_PATH, "-entry", seq_id],
        capture_output=True, text=True, timeout=30
    )
    if result.returncode != 0:
        return None
    return result.stdout.strip()

def export_blast_hits(query_sequence, df_hits, save_dir="exported_results"):
    """Saves the current BLAST hit table (CSV) and the matched sequences, retrieved straight
    from the database, as a multi-FASTA file (also including the original query sequence for
    reference) to save_dir."""
    if df_hits is None or df_hits.empty:
        print("❌ Nothing to export -- run a BLAST search with results first!")
        return

    os.makedirs(save_dir, exist_ok=True)
    df_hits.to_csv(os.path.join(save_dir, "blast_hits.csv"), index=False)
    print(f"💾 [SAVED] BLAST hits table exported to '{save_dir}/blast_hits.csv'")

    fasta_records = [_to_fasta(query_sequence).replace(">user_query", ">query_sequence", 1)]
    for seq_id in df_hits["Transcript_Hit"]:
        record = get_sequence_from_db(seq_id)
        if record:
            fasta_records.append(record)

    with open(os.path.join(save_dir, "blast_hit_sequences.fasta"), "w") as f:
        f.write("\n".join(fasta_records) + "\n")
    print(f"💾 [SAVED] Query + {len(fasta_records) - 1} hit sequence(s) exported to '{save_dir}/blast_hit_sequences.fasta'")

In [ ]:
import ipywidgets as widgets
from datetime import datetime
from IPython.display import display, clear_output

def _make_browse_handler(target_box):
    """Opens a native OS folder-picker (tkinter, ships with Python) so the user can point at an
    existing folder directly instead of typing/remembering a path -- falls back to leaving the
    text box untouched if the dialog is cancelled."""
    def _browse(b):
        import tkinter as tk
        from tkinter import filedialog
        root = tk.Tk()
        root.withdraw()
        root.attributes("-topmost", True)
        folder = filedialog.askdirectory(title="Choose a folder to save results into")
        root.destroy()
        if folder:
            target_box.value = folder
    return _browse

blast_query_box = widgets.Textarea(
    placeholder="Paste a raw nucleotide sequence here (with or without a '>' FASTA header)...",
    description="Sequence:",
    layout=widgets.Layout(width="600px", height="120px")
)
blast_button = widgets.Button(description="🔬 Run BLAST Search", button_style="warning", layout=widgets.Layout(width="270px", height="40px"))
blast_export_button = widgets.Button(description="💾 Export Hit Sequences", button_style="info", layout=widgets.Layout(width="270px", height="40px"))
blast_save_location_box = widgets.Text(placeholder="exported_results (default if left blank)", description="Save to:", layout=widgets.Layout(width="360px"))
blast_browse_button = widgets.Button(description="📁 Browse...", layout=widgets.Layout(width="120px"))
blast_browse_button.on_click(_make_browse_handler(blast_save_location_box))
blast_output = widgets.Output()

def on_blast_clicked(b):
    global _CURRENT_BLAST_QUERY, _CURRENT_BLAST_HITS
    with blast_output:
        clear_output(wait=True)
        raw_sequence = blast_query_box.value.strip()
        if not raw_sequence:
            print("❌ Paste a sequence first!")
            return

        print("🔬 Running BLAST search against the HAMN01 transcriptome (transcriptome-mapped genes only)...")
        df_hits = run_blast_search(raw_sequence)
        _CURRENT_BLAST_QUERY = raw_sequence
        _CURRENT_BLAST_HITS = df_hits
        if df_hits is None:
            return

        print(f"\n✅ {len(df_hits)} hit(s) found (best match first):")
        display(df_hits)

        best_ids = df_hits.loc[df_hits["In_Dashboard"], "Gene_ID"].dropna().unique().tolist()
        if best_ids:
            print(f"\n📋 Copy into the 'Paste Box' above to view fold-change profiles: {', '.join(best_ids)}")
        else:
            print("\n⚠️ None of the matched transcripts have a corresponding entry in this dashboard's DE tables.")

def on_blast_export_clicked(b):
    with blast_output:
        clear_output(wait=True)
        _base_dir = blast_save_location_box.value.strip() or "exported_results"
        save_dir = os.path.join(_base_dir, datetime.now().strftime("search_%Y-%m-%d_%H-%M-%S"))
        print(f"📁 Saving this BLAST search to its own new folder: {save_dir}/")
        export_blast_hits(_CURRENT_BLAST_QUERY, _CURRENT_BLAST_HITS, save_dir=save_dir)

blast_button.on_click(on_blast_clicked)
blast_export_button.on_click(on_blast_export_clicked)

print("🔬 BLAST Sequence Search Ready:")
display(widgets.VBox([
    widgets.HTML("<i>Paste a sequence of interest below to find its matching transcript ID(s) among the transcriptome-mapped genes.</i>"),
    blast_query_box,
    widgets.HBox([blast_save_location_box, blast_browse_button]),
    widgets.HBox([blast_button, blast_export_button]),
    blast_output
]))

In [ ]:
import re
from IPython.display import display, HTML
from matplotlib.patches import Patch

# Global tracking variables so the export button knows what is currently active
_CURRENT_GENES = []
_CURRENT_COMPARISONS = []
_CURRENT_RESOLVED_TERMS = []  # all resolved gene IDs (incl. ones not found), for annotation printing

NOTSIG_COLOR = "#95a5a6"
DEFAULT_UP_COLOR = "#c0392b"
DEFAULT_DOWN_COLOR = "#2980b9"

def _safe_filename(gene_id):
    """Strips characters Windows forbids in filenames, so no gene ID can ever crash plt.savefig."""
    return re.sub(r'[\\/:*?"<>|]', "_", gene_id)

def _sig_colors(cmap="RdBu_r"):
    """Default keeps this dashboard's original fixed red/blue significance colors (independent
    of the heatmap's red-blue default). Non-default (colorblind) choices sample the up/down
    colors from the ends of the same palette picked for the heatmap instead."""
    if cmap == "RdBu_r":
        return DEFAULT_UP_COLOR, DEFAULT_DOWN_COLOR
    cmap_obj = plt.get_cmap(cmap)
    return cmap_obj(0.85), cmap_obj(0.15)

def _resolve_search_term(term):
    """Expand a curated gene SYMBOL into its underlying gene ID(s); otherwise treat the term as a
    literal gene/transcript ID. Accepts symbols with or without the picker's "★ " marker prefix."""
    term = term.strip()
    if term.startswith("★"):
        term = term.lstrip("★").strip()
    if term in SYMBOL_TO_IDS:
        return SYMBOL_TO_IDS[term]
    return [term]

def _lookup_annotation(gene_id):
    """Check the curated gene-of-interest summary first, then fall back to the DE tables\' own
    parsed BLAST best-hit description."""
    rows = []
    summary_hits = df_summary[df_summary["gene_id"] == gene_id]
    for _, r in summary_hits.iterrows():
        category, mapping, de_analysis = r["category"], r["mapping"], r["de_analysis"]
        source = f"Curated summary ({category}, {mapping}, DE analysis {int(de_analysis)})"
        rows.append({"Gene_ID": gene_id, "Source": source, "Symbol_or_Description": r["gene name"]})

    de_hits = df_de[(df_de["gene_id"] == gene_id) & df_de["best_hit_description"].notna()]
    if not de_hits.empty:
        rows.append({"Gene_ID": gene_id, "Source": "BLAST best hit",
                      "Symbol_or_Description": de_hits.iloc[0]["best_hit_description"]})

    return rows

def _comparison_table(gene_id, comparisons_chosen):
    """One row per chosen comparison: log2FC, padj, baseMean (DESeq2's overall normalized expression
    level for that comparison, not split by genotype), and a significance status derived from
    padj < 0.05. Comparisons the gene was never tested in are marked "Not tested" rather than plotted
    as zero -- this happens legitimately and often, since genome-mapped and transcriptome-mapped genes
    are two entirely separate ID spaces, and Dataset S1 only covers a small curated subset of genes
    compared to S3/S4. Looks each (gene_id, comparison_label) pair up in GENE_COMPARISON_INDEX (built
    once at load time) instead of re-scanning df_de's 130K rows with a fresh boolean mask per lookup."""
    rows = []
    for label in comparisons_chosen:
        hit = GENE_COMPARISON_INDEX.get((gene_id, label))
        abbr = LABEL_TO_ABBR.get(label, label)
        comp_type = LABEL_TO_COMPARISON.get(label, "")
        if hit is None:
            rows.append({"Comparison": label, "Abbr": abbr, "Type": comp_type,
                         "log2FC": np.nan, "padj": np.nan, "baseMean": np.nan, "Status": "Not tested"})
            continue

        logfc, padj, base_mean = hit["log2FoldChange"], hit["padj"], hit["baseMean"]
        if pd.notna(padj) and padj < 0.05:
            status = "Significant (up)" if logfc > 0 else "Significant (down)"
        else:
            status = "Not significant"
        rows.append({"Comparison": label, "Abbr": abbr, "Type": comp_type,
                     "log2FC": logfc, "padj": padj, "baseMean": base_mean, "Status": status})

    return pd.DataFrame(rows)

def _format_for_display(table):
    """Copy of a comparison table with NaN log2FC/padj/baseMean replaced by "-" for untested
    comparisons, so the printed table reads as "Not tested" rather than a literal NaN."""
    display_table = table.copy()
    display_table["log2FC"] = display_table["log2FC"].apply(lambda v: f"{v:.3f}" if pd.notna(v) else "-")
    display_table["padj"] = display_table["padj"].apply(lambda v: f"{v:.2e}" if pd.notna(v) else "-")
    display_table["baseMean"] = display_table["baseMean"].apply(lambda v: f"{v:.1f}" if pd.notna(v) else "-")
    return display_table

def _status_color(status, cmap="RdBu_r"):
    up_color, down_color = _sig_colors(cmap)
    if status == "Significant (up)":
        return up_color
    if status == "Significant (down)":
        return down_color
    return NOTSIG_COLOR

def _render_bar_chart(gene_id, table, cmap="RdBu_r"):
    # Use short abbreviated codes on the x-axis (see ABBREVIATION_LEGEND for what they mean) instead of
    # the full "S3 | nr0b+/- vs nr0b+/+ | immature heads | transcriptome | DE analysis 2"-style labels,
    # which were too long to fit on the chart. Only prefix each tick with its comparison-type abbreviation
    # if the table actually spans more than one genotype comparison (normally it won't, since the
    # dashboard's Pairwise dropdown scopes every plot to a single comparison already).
    if table["Type"].nunique() > 1:
        tick_labels = [f"{COMPARISON_ABBR.get(t, t)}:{a}" for t, a in zip(table["Type"], table["Abbr"])]
    else:
        tick_labels = table["Abbr"].tolist()

    up_color, down_color = _sig_colors(cmap)
    fig, ax = plt.subplots(figsize=(max(7, len(table) * 0.8), 4.5))
    colors = [_status_color(s, cmap) for s in table["Status"]]
    ax.bar(tick_labels, table["log2FC"], color=colors, width=0.6)
    ax.axhline(0, color="black", linewidth=0.8)

    legend_elements = [
        Patch(facecolor=up_color, label="Significant, upregulated (padj<0.05)"),
        Patch(facecolor=down_color, label="Significant, downregulated (padj<0.05)"),
        Patch(facecolor=NOTSIG_COLOR, label="Not significant"),
    ]
    ax.legend(handles=legend_elements, fontsize=10, loc="best")

    plt.title(f"log2 Fold Change: {gene_id}", fontsize=16, fontweight="bold", pad=12)
    plt.ylabel("log2 Fold Change", fontsize=14, fontweight="bold")
    plt.xlabel("Comparison (see abbreviation key above)", fontsize=12, fontweight="bold")
    plt.xticks(rotation=45, ha="right", fontsize=11)
    plt.yticks(fontsize=13)
def _render_basemean_chart(gene_id, table, cmap="RdBu_r"):
    """Overall expression-level chart: one bar per comparison showing DESeq2's baseMean (the
    average normalized read count across ALL samples in that comparison). This is NOT split by
    genotype -- the paper's released tables only ever report one baseMean per comparison, not a
    separate value per condition, so this chart answers "how highly expressed is this gene
    overall in this experiment?" rather than "which genotype expresses it more" (that question
    is answered by the log2 fold-change chart above)."""
    if table["Type"].nunique() > 1:
        tick_labels = [f"{COMPARISON_ABBR.get(t, t)}:{a}" for t, a in zip(table["Type"], table["Abbr"])]
    else:
        tick_labels = table["Abbr"].tolist()

    cmap_obj = plt.get_cmap(cmap if cmap != "RdBu_r" else "mako")
    bar_color = cmap_obj(0.55)

    fig, ax = plt.subplots(figsize=(max(7, len(table) * 0.8), 4.5))
    ax.bar(tick_labels, table["baseMean"], color=bar_color, width=0.6)

    plt.title(f"Overall Expression Level (baseMean): {gene_id}", fontsize=16, fontweight="bold", pad=12)
    plt.ylabel("Mean Normalized Expression\n(baseMean, DESeq2 -- not split by genotype)", fontsize=13, fontweight="bold")
    plt.xlabel("Comparison (see abbreviation key above)", fontsize=12, fontweight="bold")
    plt.xticks(rotation=45, ha="right", fontsize=11)
    plt.yticks(fontsize=13)
    sns.despine()
    plt.tight_layout()
    return fig

def plot_multi_genes(search_terms, comparisons_chosen=None):
    """Resolves search terms (raw IDs or symbols) into gene IDs and stores the result in
    _CURRENT_GENES/_CURRENT_COMPARISONS/_CURRENT_RESOLVED_TERMS. Call _render_bar_charts()/
    _print_annotations() afterward to actually display output -- split out so the caller
    controls display order (heatmap, then plots, then annotations)."""
    global _CURRENT_GENES, _CURRENT_COMPARISONS, _CURRENT_RESOLVED_TERMS

    comparisons = comparisons_chosen if comparisons_chosen else COMPARISON_OPTIONS
    _CURRENT_COMPARISONS = comparisons

    gene_ids = []
    for term in search_terms:
        gene_ids.extend(_resolve_search_term(term))
    gene_ids = sorted(set(gene_ids))
    _CURRENT_RESOLVED_TERMS = gene_ids
    _CURRENT_GENES = [g for g in gene_ids if g in VALID_GENE_IDS]

def _render_bar_charts(cmap="RdBu_r"):
    if _CURRENT_GENES:
        print("\n📖 ABBREVIATION KEY (used on the chart x-axes below):")
        display(ABBREVIATION_LEGEND)

    for gene_id in _CURRENT_GENES:
        table = _comparison_table(gene_id, _CURRENT_COMPARISONS).dropna(subset=["log2FC"])
        if table.empty:
            print(f"\n(No comparisons with data to plot for {gene_id} among the selected comparisons.)")
            continue
        _render_bar_chart(gene_id, table, cmap)
        plt.show()
        _render_basemean_chart(gene_id, table, cmap)
        plt.show()

def _print_annotations(gene_ids=None):
    if gene_ids is None:
        gene_ids = _CURRENT_RESOLVED_TERMS
    print("\n📋 GENE ANNOTATION & COMPARISON STATUS:")
    for gene_id in gene_ids:
        rows = _lookup_annotation(gene_id)
        if rows:
            print(f"\nAnnotations for Search ID: {gene_id}")
            display(pd.DataFrame(rows))
        else:
            print(f"\nAnnotations for Search ID: {gene_id} -> 🛑 [No annotation found]")

        if gene_id in VALID_GENE_IDS:
            print(f"Comparison status for {gene_id}:")
            display(_format_for_display(_comparison_table(gene_id, _CURRENT_COMPARISONS)))
        else:
            print(f"🛑 {gene_id} was not found in any DE comparison table.")

def _keyword_regex(keyword):
    """Left-word-boundary pattern: the keyword must start at a word boundary (so a short keyword
    can't silently match as a fragment buried inside an unrelated word -- e.g. "vasa" matching
    "extravasation"), but is still free to be a PREFIX of a longer word (so "kinase" still
    matches "kinases", "kinase-like", etc.)."""
    return r"\b" + re.escape(keyword)

def _search_annotations_by_keyword(keyword):
    """Finds every gene whose curated gene-of-interest name or BLAST best-hit description
    contains the given keyword as a word-initial match (see _keyword_regex), returning gene IDs
    actually present in this dashboard's DE tables (i.e. in VALID_GENE_IDS)."""
    keyword = keyword.strip()
    if not keyword:
        return []
    pattern = _keyword_regex(keyword)
    matched = set()

    summary_mask = df_summary["gene name"].astype(str).str.contains(pattern, case=False, na=False, regex=True)
    matched.update(df_summary.loc[summary_mask, "gene_id"].astype(str))

    de_mask = df_de["best_hit_description"].astype(str).str.contains(pattern, case=False, na=False, regex=True)
    matched.update(df_de.loc[de_mask, "gene_id"].astype(str))

    matched_ids = [g for g in matched if g in VALID_GENE_IDS]
    return sorted(matched_ids)

# --- SEPARATE EXPORT EXECUTION FUNCTION ---
def export_cached_results(save_dir="exported_results", cmap="RdBu_r"):
    if not _CURRENT_GENES:
        print("❌ Nothing available to save. Please run a successful query search first!")
        return

    os.makedirs(save_dir, exist_ok=True)

    all_rows = []
    for gene_id in _CURRENT_GENES:
        ann_rows = _lookup_annotation(gene_id) or [{"Gene_ID": gene_id, "Source": "None", "Symbol_or_Description": ""}]
        comp_table = _comparison_table(gene_id, _CURRENT_COMPARISONS)
        for base_row in ann_rows:
            merged = dict(base_row)
            for _, comp_row in comp_table.iterrows():
                merged[comp_row["Comparison"]] = comp_row["Status"]
            all_rows.append(merged)

    if all_rows:
        pd.DataFrame(all_rows).to_csv(os.path.join(save_dir, "selected_gene_annotations.csv"), index=False)
        print(f"💾 [SAVED] Annotation + comparison-status CSV created inside '{save_dir}/' folder.")

    for gene_id in _CURRENT_GENES:
        table = _comparison_table(gene_id, _CURRENT_COMPARISONS).dropna(subset=["log2FC"])
        if table.empty:
            continue
        _render_bar_chart(gene_id, table, cmap)
        plt.savefig(os.path.join(save_dir, f"log2fc_profile_{_safe_filename(gene_id)}.pdf"), format="pdf", dpi=300, bbox_inches="tight")
        plt.close()

        _render_basemean_chart(gene_id, table, cmap)
        plt.savefig(os.path.join(save_dir, f"basemean_profile_{_safe_filename(gene_id)}.pdf"), format="pdf", dpi=300, bbox_inches="tight")
        plt.close()

    print(f"💾 [SAVED] Vector PDF plots for {_CURRENT_GENES} generated inside '{save_dir}/' folder.")

In [ ]:
from scipy.cluster.hierarchy import linkage

# --- Multi-gene comparison view: hierarchically-clustered heatmap (Aim E instructions: "generate an
# interactive heatmap..."), adapted for this dataset's log2FC-per-comparison shape instead of a single
# expression matrix. Only meaningful with 2+ genes. log2FC is already a centered, comparable ratio, so
# no additional z-scoring is applied here -- clustering runs directly on the log2FC values themselves.

CMAP_OPTIONS = {
    "Default": "RdBu_r",
    "Purple-Orange (colorblind-safe)": "PuOr",
    "Brown-Teal (colorblind-safe)": "BrBG",
    "Cividis (colorblind-safe, single-hue)": "cividis",
}

def _build_logfc_matrices(gene_ids, comparisons_chosen):
    cols_abbr = [LABEL_TO_ABBR.get(c, c) for c in comparisons_chosen]
    logfc_mat = pd.DataFrame(index=gene_ids, columns=cols_abbr, dtype=float)
    padj_mat = pd.DataFrame(index=gene_ids, columns=cols_abbr, dtype=float)
    for gene_id in gene_ids:
        table = _comparison_table(gene_id, comparisons_chosen)
        for col, (_, row) in zip(cols_abbr, table.iterrows()):
            logfc_mat.loc[gene_id, col] = row["log2FC"]
            padj_mat.loc[gene_id, col] = row["padj"]
    return logfc_mat, padj_mat

def _render_heatmap(gene_ids, comparisons_chosen, cmap="RdBu_r"):
    logfc_mat, _ = _build_logfc_matrices(gene_ids, comparisons_chosen)

    # Drop any gene/comparison with zero valid data among the current selection -- otherwise a
    # fully-blank row or column (very common here, since most genes are only tested in a handful
    # of the 20 pairwise comparisons) pads the heatmap out with wasted, confusing empty space.
    logfc_mat = logfc_mat.loc[logfc_mat.notna().any(axis=1), logfc_mat.notna().any(axis=0)]

    finite = logfc_mat.values[np.isfinite(logfc_mat.values.astype(float))]
    vabs = float(np.abs(finite).max()) if finite.size else 1.0
    vabs = vabs or 1.0

    # Hierarchical clustering (genes and comparisons): NaN ("not tested") is filled with 0 only for
    # the clustering-distance calculation (scipy's linkage can't handle NaN); the actual displayed
    # cell still renders blank via mask=, so "not tested" is never disguised as "no change".
    logfc_fill = logfc_mat.fillna(0).astype(float)
    n_genes, n_cols = logfc_mat.shape
    row_linkage = linkage(logfc_fill, method="average") if n_genes > 1 else None
    col_linkage = linkage(logfc_fill.T, method="average") if n_cols > 1 else None

    g = sns.clustermap(
        logfc_mat.astype(float), cmap=cmap, center=0, vmin=-vabs, vmax=vabs, mask=logfc_mat.isna(),
        row_linkage=row_linkage, col_linkage=col_linkage,
        row_cluster=n_genes > 1, col_cluster=n_cols > 1,
        linewidths=0.5, annot=False,
        figsize=(max(6, n_cols * 0.9) + 2.2, max(4, n_genes * 1.3 + 2)),
        cbar_pos=(0.90, 0.86, 0.025, 0.10), dendrogram_ratio=(0.15, 0.15),
    )

    # sns.clustermap lets the heatmap/col-dendrogram extend almost to the figure's right edge
    # regardless of cbar_pos (the colorbar floats independently, on top), so it silently overlaps
    # the heatmap's top-right corner. Directly shrink the heatmap + column-dendrogram axes to end
    # before the colorbar's left edge -- NOT via fig.subplots_adjust(), which was found (by direct
    # testing) to trigger an internal seaborn recompute that snaps cbar_pos back to its own default
    # top-left position instead of respecting the value passed in. The colorbar is also placed
    # above the heatmap (not beside its top rows) so it never collides with row labels either.
    cbar_left, margin = 0.90, 0.02
    for ax in (g.ax_heatmap, g.ax_col_dendrogram):
        pos = ax.get_position()
        ax.set_position([pos.x0, pos.y0, (cbar_left - margin) - pos.x0, pos.height])

    g.ax_heatmap.set_xlabel("Comparison (see abbreviation key above)", fontsize=16, fontweight="bold")
    g.ax_heatmap.set_ylabel("")
    plt.setp(g.ax_heatmap.get_xticklabels(), rotation=30, ha="right", fontsize=14)
    plt.setp(g.ax_heatmap.get_yticklabels(), rotation=-30, ha="left", va="center", fontsize=13)
    g.ax_cbar.set_title("log2FC", fontsize=11, fontweight="bold")
    g.ax_cbar.tick_params(labelsize=8)
    g.fig.suptitle("log2 Fold Change Heatmap (blank = not tested, hierarchically clustered)",
                   fontsize=17, fontweight="bold", x=0.42, ha="center", y=1.08)
    return g.fig

def _has_any_valid_data(gene_ids, comparisons_chosen):
    logfc_mat, _ = _build_logfc_matrices(gene_ids, comparisons_chosen)
    return bool(logfc_mat.notna().to_numpy().sum())

def render_multi_gene_views(gene_ids, comparisons_chosen, cmap="RdBu_r"):
    """Call after plot_multi_genes(): shows the clustered heatmap when 2+ genes are selected."""
    if len(gene_ids) < 2:
        return
    if not _has_any_valid_data(gene_ids, comparisons_chosen):
        print("\n⚠️ None of the selected genes were tested in any of the chosen comparisons -- heatmap skipped.")
        return
    print("\n📊 MULTI-GENE COMPARISON HEATMAP:")
    _render_heatmap(gene_ids, comparisons_chosen, cmap)
    plt.show()

def save_multi_gene_views(gene_ids, comparisons_chosen, cmap="RdBu_r", save_dir="exported_results"):
    """Call after export_cached_results(): saves the heatmap as a PDF when 2+ genes are selected."""
    if len(gene_ids) < 2:
        return
    if not _has_any_valid_data(gene_ids, comparisons_chosen):
        print("\n⚠️ None of the selected genes were tested in any of the chosen comparisons -- heatmap skipped.")
        return
    os.makedirs(save_dir, exist_ok=True)
    fig = _render_heatmap(gene_ids, comparisons_chosen, cmap)
    fig.savefig(os.path.join(save_dir, "multi_gene_heatmap.pdf"), format="pdf", dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"💾 [SAVED] Heatmap PDF generated inside '{save_dir}/' folder.")


In [ ]:
import ipywidgets as widgets
from datetime import datetime
from IPython.display import display, clear_output

def _make_browse_handler(target_box):
    """Opens a native OS folder-picker (tkinter, ships with Python) so the user can point at an
    existing folder directly instead of typing/remembering a path. Defined locally (not reused
    from the BLAST cell above) so this cell stays self-sufficient regardless of cell order."""
    def _browse(b):
        import tkinter as tk
        from tkinter import filedialog
        root = tk.Tk()
        root.withdraw()
        root.attributes("-topmost", True)
        folder = filedialog.askdirectory(title="Choose a folder to save results into")
        root.destroy()
        if folder:
            target_box.value = folder
    return _browse

# Glossary Accordion View
key_details = widgets.HTML(value='''
<div style="background-color: #f7f9fa; padding: 12px; border-left: 4px solid #2b5b84; border-radius: 4px; font-family: sans-serif; font-size: 13px; line-height: 1.5;">
    <b>📋 HOW TO READ THIS DASHBOARD:</b><br>
    Each bar shows a gene\'s <b>log2 fold change</b> in one specific pairwise comparison
    (e.g. lcry-/- vs lcry+/+ in immature heads). There is no single expression matrix here --
    bars are omitted entirely for comparisons the gene wasn\'t tested in.<br>
    Bars are color-coded by significance (padj &lt; 0.05): red = upregulated, blue = downregulated,
    gray = not significant by default -- choosing a colorblind-safe <b>Palette</b> below recolors
    upregulated/downregulated using opposite ends of that palette instead. See the legend on each
    chart for the exact colors.<br><br>
    Pick a genotype comparison in <b>Pairwise</b> first -- the <b>Details</b> box below it refreshes to
    only the tissue/mapping/DE-analysis variants that actually exist for that comparison.<br><br>
    Search by <b>raw ID</b> (e.g. <code>XLOC_059112</code>, <code>comp2263280_c0_seq1</code>) or by a
    <b>curated gene symbol</b> marked with ★ (e.g. <code>★ cp17a1/2</code>) -- symbols expand to
    every matching gene ID automatically.<br><br>
    <b>📖 THE 4 PAIRWISE COMPARISONS:</b><br>
    <table style="border-collapse:collapse; margin-top:4px;">
    <tr><td style="padding:2px 10px 2px 0;"><b>lcry-/- vs lcry+/+</b></td><td>l-cry homozygous mutant (both copies disrupted) vs. its wild-type sibling control.</td></tr>
    <tr><td style="padding:2px 10px 2px 0;"><b>nr0b+/- vs nr0b+/+</b></td><td>nr0b1/2 heterozygous mutant (one copy disrupted) vs. its wild-type sibling control.</td></tr>
    <tr><td style="padding:2px 10px 2px 0;"><b>nr0b-/- vs nr0b+/+</b></td><td>nr0b1/2 homozygous mutant (both copies disrupted) vs. its wild-type sibling control.</td></tr>
    <tr><td style="padding:2px 10px 2px 0;"><b>nr0b-/- vs nr0b+/-</b></td><td>nr0b1/2 homozygous mutant vs. heterozygous mutant (a gene-dosage comparison).</td></tr>
    </table>
    Every comparison is a mutant vs. its own matched control from the same experiment -- rows are
    never cross-compared (e.g. an l-cry sample is never compared against an nr0b sample).<br><br>
    <b>What "Details" means once you've picked a Pairwise comparison:</b> each comparison may have
    been tested more than once, varying by <b>dataset</b> (S1 / S3 / S4, different supplementary
    tables in the paper), <b>tissue</b> (immature heads / premature female heads / immature whole
    worms), <b>mapping</b> (reads aligned to the draft genome vs. the reference transcriptome), and
    <b>DE analysis</b> (an independent repeat of the differential-expression test). Selecting more
    than one Details entry plots that gene across all of them side by side.<br><br>
    <b>Two charts per gene:</b> the <b>log2 Fold Change</b> chart shows how much the gene's
    expression differs between the mutant and its control. The <b>Overall Expression Level
    (baseMean)</b> chart shows how highly expressed the gene is overall in that comparison
    (DESeq2's baseMean, averaged across every sample) -- this is <i>not</i> split by genotype,
    since the paper's released tables only report one such value per comparison, not per group.
</div>
''')
glossary_dropdown = widgets.Accordion(children=[key_details], selected_index=None)
glossary_dropdown.set_title(0, "🔍 Click to view/hide Dashboard Legend")
glossary_dropdown.layout.width = "700px"

# Inputs
# Step 1 is now two linked boxes rather than one flat 20-item list: a single-choice "Pairwise"
# dropdown picks the genotype comparison, and the "Details" multiselect (tissue/mapping/DE analysis)
# updates to show only the combinations that actually exist for that comparison.
comparison_type_dropdown = widgets.Dropdown(
    options=COMPARISON_TYPES,
    description="Pairwise:",
    layout=widgets.Layout(width="420px")
)
cmap_dropdown = widgets.Dropdown(options=list(CMAP_OPTIONS.keys()), description="Palette:", layout=widgets.Layout(width="420px"))

def _details_for(comp_type):
    return sorted(COMPARISON_DETAILS[comp_type].keys())

detail_multiselect = widgets.SelectMultiple(
    options=_details_for(comparison_type_dropdown.value),
    value=tuple(_details_for(comparison_type_dropdown.value)),
    description="Details:",
    layout=widgets.Layout(width="620px", height="160px")
)

def on_comparison_type_change(change):
    new_options = _details_for(change["new"])
    detail_multiselect.options = new_options
    detail_multiselect.value = tuple(new_options)
comparison_type_dropdown.observe(on_comparison_type_change, names="value")

# Filter List: precompute the sorted option list (raw IDs + "★ symbol" entries) and a lowercase
# mirror ONCE here, rather than re-lowering ~100K strings inside the keystroke handler below.
# The widget itself is also capped to MAX_FILTER_RESULTS entries -- with 100K+ gene IDs, pushing
# every match into a SelectMultiple after every keystroke is what made typing feel unresponsive.
_SEARCH_OPTIONS = sorted(df_de["gene_id"].unique().tolist()) + sorted(f"★ {sym}" for sym in SYMBOL_TO_IDS.keys())
_SEARCH_OPTIONS_LOWER = [s.lower() for s in _SEARCH_OPTIONS]
MAX_FILTER_RESULTS = 500

search_filter = widgets.Text(placeholder="Type to filter (e.g. XLOC_0059, comp226, cp17a1)...", description="Filter List:", layout=widgets.Layout(width="440px"))
gene_multiselect = widgets.SelectMultiple(options=_SEARCH_OPTIONS[:MAX_FILTER_RESULTS], description="Select Genes:", layout=widgets.Layout(width="440px", height="140px"))
filter_hint = widgets.HTML(value=f"<i style='font-size:11px;color:#666'>Showing first {MAX_FILTER_RESULTS} of {len(_SEARCH_OPTIONS)} -- type to narrow down.</i>")

def update_multiselect_options(change):
    search_term = change["new"].strip().lower()
    if search_term:
        matches = [opt for opt, opt_lower in zip(_SEARCH_OPTIONS, _SEARCH_OPTIONS_LOWER) if search_term in opt_lower]
    else:
        matches = _SEARCH_OPTIONS
    gene_multiselect.options = matches[:MAX_FILTER_RESULTS]
    if len(matches) > MAX_FILTER_RESULTS:
        filter_hint.value = f"<i style='font-size:11px;color:#666'>Showing first {MAX_FILTER_RESULTS} of {len(matches)} matches -- refine your search to narrow further.</i>"
    else:
        filter_hint.value = f"<i style='font-size:11px;color:#666'>{len(matches)} matching entr{'y' if len(matches) == 1 else 'ies'}.</i>"
search_filter.observe(update_multiselect_options, names="value")

multi_gene_input = widgets.Textarea(placeholder="OR paste multiple Gene IDs or symbols here (e.g. XLOC_059112, cp17a1/2)...", description="Paste Box:", layout=widgets.Layout(width="440px", height="100px"))

# --- Option C: Search by Annotation Keyword -- finds every gene whose curated summary/BLAST-hit
# annotation contains a keyword, then runs it through the exact same Heatmap -> Plots -> Annotation
# pipeline as the other two search options, sharing the same output_area below, scoped to whichever
# Pairwise comparison + Details are currently selected. ---
keyword_search_box = widgets.Text(placeholder="e.g. cytochrome, receptor, kinase...", description="Keyword:", layout=widgets.Layout(width="250px"))
keyword_search_button = widgets.Button(description="🔎 Search Annotations", button_style="warning", layout=widgets.Layout(width="250px", height="30px"))

MAX_KEYWORD_MATCHES = 30

# TWO SEPARATE BUTTON CONTROLS
submit_button = widgets.Button(description="🎯 Generate Visuals (View Only)", button_style="success", layout=widgets.Layout(width="330px", height="40px"))
export_button = widgets.Button(description="💾 Export Active Results (Save CSV/PDF)", button_style="info", layout=widgets.Layout(width="330px", height="40px"))
save_location_box = widgets.Text(placeholder="exported_results (default if left blank)", description="Save to:", layout=widgets.Layout(width="360px"))
browse_button = widgets.Button(description="📁 Browse...", layout=widgets.Layout(width="120px"))
browse_button.on_click(_make_browse_handler(save_location_box))

output_area = widgets.Output()

def on_submit_clicked(b):
    with output_area:
        clear_output(wait=True)
        chosen_terms = list(gene_multiselect.value)
        pasted_terms = [g.strip() for g in multi_gene_input.value.replace(",", " ").split() if g.strip()]
        final_terms = sorted(set(chosen_terms + pasted_terms))

        if not final_terms:
            print("❌ Input target genes first!")
            return

        comp_type = comparison_type_dropdown.value
        chosen_labels = COMPARISON_DETAILS[comp_type]
        chosen_comparisons = [chosen_labels[d] for d in detail_multiselect.value]
        cmap = CMAP_OPTIONS[cmap_dropdown.value]

        # Output order: Heatmap -> Plots -> Annotation
        plot_multi_genes(final_terms, comparisons_chosen=chosen_comparisons)
        render_multi_gene_views(_CURRENT_GENES, _CURRENT_COMPARISONS, cmap)
        _render_bar_charts(cmap)
        _print_annotations()

def on_export_clicked(b):
    with output_area:
        _base_dir = save_location_box.value.strip() or "exported_results"
        save_dir = os.path.join(_base_dir, datetime.now().strftime("search_%Y-%m-%d_%H-%M-%S"))
        print(f"📁 Saving this search to its own new folder: {save_dir}/")
        cmap = CMAP_OPTIONS[cmap_dropdown.value]
        export_cached_results(save_dir=save_dir, cmap=cmap)
        save_multi_gene_views(_CURRENT_GENES, _CURRENT_COMPARISONS, cmap, save_dir=save_dir)

def on_keyword_search_clicked(b):
    with output_area:
        clear_output(wait=True)
        keyword = keyword_search_box.value.strip()
        if not keyword:
            print("❌ Type a keyword first!")
            return

        matched_ids = _search_annotations_by_keyword(keyword)
        if not matched_ids:
            print(f"🛑 No genes found with '{keyword}' in their annotation.")
            return

        print(f"✅ Found {len(matched_ids)} gene(s) matching '{keyword}'.")
        if len(matched_ids) > MAX_KEYWORD_MATCHES:
            print(f"⚠️ Limiting to the first {MAX_KEYWORD_MATCHES} matches to keep the plots manageable.")
            matched_ids = matched_ids[:MAX_KEYWORD_MATCHES]

        comp_type = comparison_type_dropdown.value
        chosen_labels = COMPARISON_DETAILS[comp_type]
        chosen_comparisons = [chosen_labels[d] for d in detail_multiselect.value]
        cmap = CMAP_OPTIONS[cmap_dropdown.value]

        # Output order: Heatmap -> Plots -> Annotation
        plot_multi_genes(matched_ids, comparisons_chosen=chosen_comparisons)
        render_multi_gene_views(_CURRENT_GENES, _CURRENT_COMPARISONS, cmap)
        _render_bar_charts(cmap)
        _print_annotations()

submit_button.on_click(on_submit_clicked)
export_button.on_click(on_export_clicked)
keyword_search_button.on_click(on_keyword_search_clicked)

# Layout Setup
print("📊 Fold-Change Comparison Dual-Control Workspace Ready:")
display(widgets.VBox([
    glossary_dropdown,
    widgets.HTML("<br><b>Step 1: Choose a Pairwise Comparison, then its Details</b>"),
    widgets.HBox([comparison_type_dropdown, cmap_dropdown]),
    detail_multiselect,
    widgets.HTML("<br><b>Step 2: Choose Target Genes (by ID or ★ symbol)</b>"),
    widgets.HBox([
        widgets.VBox([widgets.HTML("<i>Option A: Filter List</i>"), search_filter, gene_multiselect, filter_hint]),
        widgets.VBox([widgets.HTML("<i>Option B: Paste Box</i>"), multi_gene_input]),
        widgets.VBox([widgets.HTML("<i>Option C: Search by Keyword</i>"), keyword_search_box, keyword_search_button]),
    ]),
    widgets.HTML("<br><b>Step 3: Run and/or Save</b>"),
    widgets.HBox([save_location_box, browse_button]),
    widgets.HBox([submit_button, export_button]),
    output_area
]))